# ⚡ Kilat: MoE Pre-Training (150M 10B)

```text
  ██╗  ██╗██╗██╗      █████╗ ████████╗
  ██║ ██╔╝██║██║     ██╔══██╗╚══██╔══╝
  █████╔╝ ██║██║     ███████║   ██║   
  ██╔═██╗ ██║██║     ██╔══██║   ██║   
  ██║  ██╗██║███████╗██║  ██║   ██║   
  ╚═╝  ╚═╝╚═╝╚══════╝╚═╝  ╚═╝   ╚═╝   
  ────────────────────────────────────────────────────────────
  ⚡ KILAT | Kernelized Lightweight Attention Transformer (v2.0)
  Architecture: Hybrid Sub-Quadratic Global Decay + Latent MLA
  ────────────────────────────────────────────────────────────
```

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Airukua/Kilat/blob/main/notebooks/kilat_moe_pretraining_colab.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Airukua%2FKilat-181717?style=flat-square&logo=github)](https://github.com/Airukua/Kilat)
[![Hugging Face](https://img.shields.io/badge/🤗%20HuggingFace-AiRukua%2F150M--Kilat--SMALL--10B-FFD21E?style=flat-square)](https://huggingface.co/AiRukua/150M-Kilat-SMALL-10B)

Welcome to the **Kilat MoE Pre-Training** guide! In this interactive notebook, you will learn how to:
1. Initialize the **Kilat v2.0** hybrid architecture: **Global Decay ($O(1)$ recurrent scan)** + **Latent Multi-Head Attention (MLA)** + **DeepSeek-style Mixture-of-Experts (MoE)**.
2. Configure **150M parameters** (~151.0M total, ~59.5M active per token) with 8 routed experts (Top-2 active) + 1 shared expert.
3. Stream pre-training data from FineWeb-Edu (`gpjt/fineweb-edu-gpt2-tokens`).
4. Resume training seamlessly from any intermediate or final checkpoint without losing optimizer state or RNG determinism.
5. Export trained weights to Hugging Face format (`safetensors` + standalone remote code) and run autoregressive inference.

## 1. Hardware & Environment Check
Kilat automatically detects GPU acceleration. On Google Colab, make sure you are running on a **T4, V100, or A100 GPU** runtime (Menu: *Runtime > Change runtime type > T4 GPU*).

In [ ]:
# Check GPU availability and hardware specs
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    print(f"Device Memory:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"BF16 Supported:  {torch.cuda.is_bf16_supported()}")

## 2. Clone Repository & Install Kilat
Clone the official Kilat repository and install in editable mode with development & integration dependencies.

In [ ]:
# Clone repository and install Kilat
!rm -rf Kilat
!git clone https://github.com/Airukua/Kilat.git
%cd Kilat
!pip install -q -e ".[integrations,reporting]"
!pip install -q safetensors huggingface_hub

## 3. Display Kilat ASCII Banner & Import Core Modules
Verify that the package is correctly installed by importing the architecture and displaying the official ASCII banner.

In [ ]:
from kilat.utils.banner import print_kilat_banner
print_kilat_banner()

import os
import json
from pathlib import Path
import torch
from kilat.configs.model_config import KilatConfig
from kilat.models.transformer import KilatTransformer
from kilat.engine.trainer import KilatTrainer
from kilat.engine.args import TrainingArguments
from kilat.inference.export_hf import convert_checkpoint_to_hf
from transformers import AutoTokenizer

## 4. Model Architecture & MoE Topology Configuration
Configure the 150M parameter DeepSeekMoE hybrid model:
- **`d_model`**: 384
- **`num_hidden_layers`**: 12
- **`num_attention_heads`**: 6
- **`latent_dim`**: 96 (Latent MLA compression)
- **`num_routed_experts`**: 8 (with Top-2 routing)
- **`num_shared_experts`**: 1 (always active, captures universal linguistic priors)
- **`vocab_size`**: 50,257 (GPT-2 tokenizer)
- **`max_position_embeddings`**: 2,048

In [ ]:
# Define Kilat MoE 150M Configuration
config = KilatConfig(
    vocab_size=50257,
    d_model=384,
    num_layers=12,
    num_heads=6,
    latent_dim=96,
    max_position_embeddings=2048,
    ffn_mode="moe",
    num_routed_experts=8,
    num_experts_per_tok=2,
    num_shared_experts=1,
    moe_aux_loss_coeff=0.01,
    use_bias=False,
    dropout=0.0
)

# Instantiate model
model = KilatTransformer(config)

# Calculate total and active parameter count
total_params = sum(p.numel() for p in model.parameters())
shared_expert_params = sum(p.numel() for p in model.layers[0].ffn.shared_expert.parameters()) if hasattr(model.layers[0].ffn, 'shared_expert') else 0
routed_expert_params = sum(p.numel() for p in model.layers[0].ffn.routed_experts[0].parameters()) if hasattr(model.layers[0].ffn, 'routed_experts') else 0

# Base non-FFN parameters per layer + embeddings
base_params = total_params - (12 * (shared_expert_params + 8 * routed_expert_params))
# Active parameters per token: Base + shared expert + 2 active routed experts
active_params = base_params + (12 * (shared_expert_params + 2 * routed_expert_params))

print(f"⚡ Total Parameters:  {total_params / 1e6:.2f}M")
print(f"⚡ Active Parameters: {active_params / 1e6:.2f}M per token")
print(f"⚡ Sparsity Ratio:    {100.0 * (1.0 - active_params / total_params):.1f}% compute savings per forward pass")

## 5. Seamless Checkpoint Resumption
Kilat strictly guarantees deterministic checkpoint recovery (Invariant 4):
- Weights, optimizer states, LR scheduler, AMP scaler, and all RNG seeds (PyTorch, CUDA, NumPy, Python) are restored.
- Here is how you can pull a pre-trained checkpoint from Hugging Face Hub (e.g. `AiRukua/150M-Kilat-SMALL-10B`) and resume training or run inference.

In [ ]:
from huggingface_hub import snapshot_download

checkpoint_dir = Path("./checkpoints/moe_150m_10b")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print("📥 Mengunduh checkpoint pre-trained dari Hugging Face Hub...")
snapshot_download(
    repo_id="AiRukua/150M-Kilat-SMALL-10B",
    local_dir=str(checkpoint_dir),
    ignore_patterns=["*.md", ".git*"]
)
print("✅ Checkpoint siap di:", checkpoint_dir)

## 6. Pre-Training / Resumption Loop with KilatTrainer
Configure training arguments and launch `KilatTrainer`.

In [ ]:
training_args = TrainingArguments(
    output_dir="./checkpoints/moe_150m_10b",
    learning_rate=6e-4,
    min_lr=6e-5,
    warmup_steps=500,
    max_steps=19073,           # Full 10B token budget
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    mixed_precision="fp16" if not torch.cuda.is_bf16_supported() else "bf16",
    weight_decay=0.1,
    grad_clip=1.0,
    save_steps=1000,
    eval_steps=500,
    logging_steps=50,
    seed=42
)

print("🚀 Ready to train with KilatTrainer!")
# trainer = KilatTrainer(model=model, args=training_args, train_dataset=train_dataset)
# trainer.train(resume_from_checkpoint="./checkpoints/moe_150m_10b")

## 7. Export Checkpoint to Hugging Face Safetensors Format
Convert Kilat checkpoints into a production-ready Hugging Face repository package with:
- Standalone remote code (`modeling_kilat.py`, `configuration_kilat.py`)
- Standard safetensors weights (`model.safetensors`)
- Tokenizer files & Generation configuration (`use_cache: false`, repetition penalty)

In [ ]:
hf_export_dir = Path("./checkpoints/150M-Kilat-SMALL-10B-hf")
hf_export_dir.mkdir(parents=True, exist_ok=True)

# Run official exporter
convert_checkpoint_to_hf(
    checkpoint_dir=checkpoint_dir,
    output_dir=hf_export_dir,
    use_safetensors=True,
    device="cpu",
    strict=False,
    skip_validation=False,
    skip_tokenizer=False,
)

# Save standard GPT-2 tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.save_pretrained(hf_export_dir)

print("✅ Hugging Face export complete at:", hf_export_dir)
!ls -lh {hf_export_dir}

## 8. Interactive Inference Generation
Test generation with optimal base model sampling hyperparameters:
- `temperature`: `0.75`
- `top_p`: `0.92`
- `top_k`: `50`
- `repetition_penalty`: `1.18`
- `no_repeat_ngram_size`: `3`

In [ ]:
from transformers import AutoModelForCausalLM

# Load exported model directly via transformers AutoModel
model_hf = AutoModelForCausalLM.from_pretrained(
    hf_export_dir,
    trust_remote_code=True,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
if torch.cuda.is_available():
    model_hf = model_hf.cuda()

prompt = "The future of artificial intelligence lies in efficient"
inputs = tokenizer(prompt, return_tensors="pt")
if torch.cuda.is_available():
    inputs = {k: v.cuda() for k, v in inputs.items()}

# Generate continuation
with torch.no_grad():
    outputs = model_hf.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.75,
        top_p=0.92,
        top_k=50,
        repetition_penalty=1.18,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.eos_token_id
    )

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("⚡ [PROMPT]:", prompt)
print("⚡ [OUTPUT]:\n", result)